In [ ]:
import duckdb
import pandas as pd
import pathlib

import irp.config as _config

cfg = _config.load()
_ROOT = pathlib.Path(_config.__file__).parents[2]
DB = str((_ROOT / cfg['store']['db_path']).resolve())


def q(sql, params=None):
    with duckdb.connect(DB, read_only=True) as con:
        return con.execute(sql, params or []).df()


print('DB:', DB)

DB: /mnt/Dev/active_python_projects/investment_research_platform/data/irp.duckdb


In [38]:
# P/E ratio: price on first trading day >= Publish Date, divided by EPS
# Annual  : EPS = Net Income (Common) / Shares (Diluted)
# Quarterly: EPS = TTM (sum of last 4 quarters) / Shares (Diluted)
# Publish Date used for price lookup to avoid look-ahead bias

pe = q("""
WITH annual_eps AS (
    SELECT
        variant, Ticker, "Report Date", "Publish Date", "Fiscal Year", "Fiscal Period",
        "Net Income (Common)" / NULLIF("Shares (Diluted)", 0) AS eps
    FROM income
    WHERE variant = 'A'
      AND "Shares (Diluted)" > 0
      AND "Net Income (Common)" IS NOT NULL
      AND "Publish Date" IS NOT NULL
),
quarterly_ttm AS (
    SELECT
        variant, Ticker, "Report Date", "Publish Date", "Fiscal Year", "Fiscal Period",
        SUM("Net Income (Common)") OVER (
            PARTITION BY Ticker
            ORDER BY "Report Date"
            ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
        ) / NULLIF("Shares (Diluted)", 0) AS eps,
        COUNT(*) OVER (
            PARTITION BY Ticker
            ORDER BY "Report Date"
            ROWS BETWEEN 3 PRECEDING AND CURRENT ROW
        ) AS qtrs_in_window
    FROM income
    WHERE variant = 'Q'
      AND "Shares (Diluted)" > 0
      AND "Net Income (Common)" IS NOT NULL
      AND "Publish Date" IS NOT NULL
),
quarterly_eps AS (
    SELECT variant, Ticker, "Report Date", "Publish Date", "Fiscal Year", "Fiscal Period", eps
    FROM quarterly_ttm
    WHERE qtrs_in_window = 4
),
all_eps AS (
    SELECT * FROM annual_eps
    UNION ALL
    SELECT * FROM quarterly_eps
),
price_ranked AS (
    SELECT
        e.*,
        p.close,
        p.date AS price_date,
        ROW_NUMBER() OVER (
            PARTITION BY e.Ticker, e."Report Date", e.variant
            ORDER BY p.date ASC
        ) AS rn
    FROM all_eps e
    JOIN prices p ON p.ticker = e.Ticker AND p.date >= strftime(e."Publish Date", '%Y-%m-%d')
)
SELECT
    variant,
    Ticker,
    "Report Date",
    "Publish Date",
    "Fiscal Year",
    "Fiscal Period",
    eps,
    close AS price,
    price_date,
    close / NULLIF(eps, 0) AS pe_ratio
FROM price_ranked
WHERE rn = 1
ORDER BY Ticker, variant, "Report Date"
""")

print(pe.shape)
pe.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(42865, 10)


,variant,Ticker,Report Date,Publish Date,Fiscal Year,Fiscal Period,eps,price,price_date,pe_ratio
0,A,A,2020-10-31,2020-12-18,2020,FY,2.304487,117.847,2020-12-18,51.138058
1,A,A,2021-10-31,2021-12-17,2021,FY,3.941368,150.453,2021-12-17,38.172786
2,A,A,2022-10-31,2022-12-21,2022,FY,4.084691,150.090,2022-12-21,36.744522
3,A,A,2023-10-31,2023-12-20,2023,FY,4.189189,138.180,2023-12-20,32.984903
4,A,A,2024-10-31,2024-12-20,2024,FY,4.429553,134.510,2024-12-20,30.366493
5,Q,A,2021-04-30,2021-06-01,2021,Q2,3.022876,135.984,2021-06-01,44.984977
6,Q,A,2021-07-31,2021-09-01,2021,Q3,3.235294,174.120,2021-09-01,53.818909
7,Q,A,2021-10-31,2021-12-17,2021,Q4,3.941368,150.453,2021-12-17,38.172786
8,Q,A,2022-01-31,2022-03-03,2022,Q1,3.976898,136.472,2022-03-03,34.316196
9,Q,A,2022-04-30,2022-05-31,2022,Q2,4.196013,127.114,2022-05-31,30.293994


In [ ]:
# Distribution of P/E ratios (plausible range: 0 < P/E < 200)
annual = pe[pe['variant'] == 'A'].copy()
sane = annual[(annual['pe_ratio'] > 0) & (annual['pe_ratio'] < 200)]

print(f'Annual periods total : {len(annual):,}')
print(f'P/E in (0, 200)      : {len(sane):,}  ({100 * len(sane) / len(annual):.1f}%)')
print()
sane['pe_ratio'].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])

Annual periods total : 12,343
P/E in (0, 200)      : 6,812  (55.2%)



count    6812.000000
mean       30.778549
std        30.426146
min         0.000016
10%         7.072203
25%        12.594873
50%        21.631421
75%        36.025708
90%        65.050228
max       199.476955
Name: pe_ratio, dtype: float64

In [40]:
# Single ticker drill-down
tic = 'MSFT'
pe[pe['Ticker'] == tic].sort_values('Report Date')

,variant,Ticker,Report Date,Publish Date,Fiscal Year,Fiscal Period,eps,price,price_date,pe_ratio
25663,A,MSFT,2020-06-30,2020-07-30,2020,FY,5.763504,196.089,2020-07-30,34.022533
25668,Q,MSFT,2021-03-31,2021-04-27,2021,Q3,7.373305,253.805,2021-04-27,34.422147
25664,A,MSFT,2021-06-30,2021-07-29,2021,FY,8.043981,278.213,2021-07-29,34.586483
25669,Q,MSFT,2021-06-30,2021-07-29,2021,Q4,8.043981,278.213,2021-07-29,34.586483
25670,Q,MSFT,2021-09-30,2021-10-26,2022,Q1,8.985175,301.714,2021-10-26,33.579089
25671,Q,MSFT,2021-12-31,2022-01-25,2022,Q2,9.422237,281.193,2022-01-25,29.843550
25672,Q,MSFT,2022-03-31,2022-04-26,2022,Q3,9.590470,263.933,2022-04-26,27.520341
25673,Q,MSFT,2022-06-30,2022-07-28,2022,Q4,9.646950,270.610,2022-07-28,28.051354
25665,A,MSFT,2022-06-30,2022-07-28,2022,FY,9.646950,270.610,2022-07-28,28.051354
25674,Q,MSFT,2022-09-30,2022-10-25,2023,Q1,9.323848,245.921,2022-10-25,26.375484


In [ ]:
# Latest annual P/E per ticker (most recent Report Date per Ticker)
latest = (
    annual.sort_values('Report Date')
    .groupby('Ticker', as_index=False)
    .last()[['Ticker', 'Report Date', 'Fiscal Year', 'eps', 'price', 'pe_ratio']]
)
print(latest.shape)
latest.sort_values('pe_ratio').dropna(subset=['pe_ratio']).head(20)

(2913, 6)


,Ticker,Report Date,Fiscal Year,eps,price,pe_ratio
2243,RPAY,2024-12-31,2024,-0.000113,7.120,-63036.212627
1276,HUIZ,2024-12-31,2024,-0.000091,1.770,-19548.928010
1640,MCHP,2025-03-31,2024,-0.005025,56.190,-11181.810000
674,CRWD,2025-01-31,2024,-0.062272,308.860,-4959.876977
271,ATXG,2021-01-31,2021,-20.355201,98481.000,-4838.124572
1188,GWRE,2024-07-31,2024,-0.074163,172.760,-2329.457087
2149,RAMP,2025-03-31,2024,-0.012310,28.070,-2280.290934
1683,MIST,2024-12-31,2024,-0.000967,2.030,-2099.332428
184,AOUT,2025-04-30,2025,-0.006013,11.960,-1989.087792
99,AKTX,2024-12-31,2024,-0.033140,47.952,-1446.968166


In [42]:
# Tickers with no price match (income data but no prices loaded)
no_price = q("""
    SELECT DISTINCT i."Ticker"
    FROM income i
    WHERE NOT EXISTS (
        SELECT 1 FROM prices p WHERE p.ticker = i."Ticker"
    )
    ORDER BY i."Ticker"
""")
print(f'{len(no_price)} tickers in income with no matching prices')
no_price.head(20)

1626 tickers in income with no matching prices


,Ticker
0,A21
1,AAC
2,AACT
3,AAGH
4,AAIC
5,AAMC
6,AAN
7,AAWH
8,AAWW
9,ABIO


In [ ]:
import json, pathlib
import plotly.express as px
import plotly.io as pio
from IPython.display import display, clear_output
import ipywidgets as widgets


def _theme():
    cfg = pathlib.Path('~/.config/Code/User/settings.json').expanduser()
    try:
        t = json.loads(cfg.read_text()).get('workbench.colorTheme', '')
        return 'plotly_white' if 'light' in t.lower() else 'plotly_dark'
    except Exception:
        return 'plotly_dark'


# Latest annual EPS + market cap + sector per ticker
scatter_df = q("""
WITH latest AS (
    SELECT
        Ticker,
        "Report Date",
        "Publish Date",
        "Net Income (Common)" / NULLIF("Shares (Diluted)", 0) AS eps,
        "Shares (Diluted)"                                     AS shares_diluted,
        ROW_NUMBER() OVER (PARTITION BY Ticker ORDER BY "Report Date" DESC) AS rn
    FROM income
    WHERE variant = 'A'
      AND "Shares (Diluted)" > 0
      AND "Net Income (Common)" IS NOT NULL
      AND "Publish Date" IS NOT NULL
),
price_ranked AS (
    SELECT
        l.Ticker, l."Report Date", l.eps, l.shares_diluted,
        p.close AS price,
        ROW_NUMBER() OVER (
            PARTITION BY l.Ticker ORDER BY p.date ASC
        ) AS prn
    FROM latest l
    JOIN prices p ON p.ticker = l.Ticker
                  AND p.date >= strftime(l."Publish Date", '%Y-%m-%d')
    WHERE l.rn = 1
)
SELECT
    pr.Ticker,
    pr."Report Date",
    pr.eps,
    pr.price,
    pr.price * pr.shares_diluted / 1e9 AS market_cap_bn,
               pr.shares_diluted,
    COALESCE(ind.Sector, 'Unknown')     AS Sector
FROM price_ranked pr
LEFT JOIN companies c   ON c."Ticker"      = pr.Ticker
LEFT JOIN industries ind ON ind."IndustryId" = c."IndustryId"
WHERE prn = 1
  AND price > 0
  AND eps   > 0
""")

print(scatter_df.shape)
scatter_df.head()

(1692, 7)


,Ticker,Report Date,eps,price,market_cap_bn,shares_diluted,Sector
0,INSW,2024-12-31,8.388143,33.75,1.676704,49680127.0,Industrials
1,MSCI,2024-12-31,14.046707,579.39,45.748634,78960000.0,Financial Services
2,EQH,2024-12-31,3.694581,53.30,17.311840,324800000.0,Financial Services
3,GENC,2024-09-30,0.993178,15.35,0.225000,14658000.0,Industrials
4,HBB,2024-12-31,2.202893,17.90,0.249938,13963000.0,Consumer Cyclical


In [49]:
out = widgets.Output()
display(out)

with out:
    clear_output(wait=True)
    fig = px.scatter(
        scatter_df,
        x='market_cap_bn',
        y='eps',
        color='Sector',
        hover_name='Ticker',
        hover_data={
            'Report Date': True,
            'Sector': True,
            'price': ':.2f',
            'market_cap_bn': ':.1f',
            'eps': ':.2f',
        },
        log_x=True,
        log_y=True,
        labels={'market_cap_bn': 'Market Cap (USD bn)', 'eps': 'EPS (USD, annual)'},
        title='EPS vs Market Cap — latest annual, positive EPS only',
        template=_theme(),
    )
    fig.update_traces(marker=dict(size=5, opacity=0.7))
    fig.update_layout(height=650, legend_title_text='Sector')
    display(fig)

Output()

In [ ]:
fig = px.scatter(
    scatter_df,
    x='market_cap_bn',
    y='eps',
    color='Sector',
    hover_name='Ticker',
    hover_data={
        'Report Date': True,
        'Sector': True,
        'price': ':.2f',
        'market_cap_bn': ':.1f',
        'eps': ':.2f',
    },
    log_x=True,
    log_y=True,
    labels={'market_cap_bn': 'Market Cap (USD bn)', 'eps': 'EPS (USD, annual)'},
    title='EPS vs Market Cap — latest annual, positive EPS only',
    template=_theme(),
)
fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.update_layout(height=650, legend_title_text='Sector')
display(fig)

In [ ]:
def f(a=3, b=4):
    return a + b

In [ ]:
scatter_df[scatter_df['Ticker'] == 'SYNA']

,Ticker,Report Date,eps,price,market_cap_bn,shares_diluted,Sector
229,SYNA,2024-06-30,3140000.0,80.54,0.000003,40.0,Technology


In [46]:
mask = scatter_df['shares_diluted'] < 100000
scatter_df[mask]

,Ticker,Report Date,eps,price,market_cap_bn,shares_diluted,Sector
116,APVO,2024-12-31,3.141927e+04,1202.400,0.000923,768.0,Healthcare
229,SYNA,2024-06-30,3.140000e+06,80.540,0.000003,40.0,Technology
285,BMI,2024-12-31,4.230446e+03,215.114,0.006353,29534.0,Technology
450,CAMT,2022-12-31,1.657696e+03,27.320,0.001318,48229.0,Technology
682,DDS,2024-01-31,4.473252e+04,474.964,0.007845,16517.0,Consumer Cyclical
740,BRKR,2024-12-31,7.540000e+05,46.735,0.000007,150.0,Healthcare
759,HESM,2024-12-31,2.506742e+06,41.140,0.000004,89.0,Energy
1156,ALTI,2021-12-31,5.662737e+02,9.850,0.000069,6956.0,Financial Services
1457,SWBI,2025-04-30,2.987848e+02,8.725,0.000392,44932.0,Industrials
1515,HUBG,2024-12-31,1.701902e+03,42.180,0.002577,61104.0,Industrials
